# DLM on Colab

Thin runner: this notebook only launches the scripts in the repo (`run.py`). All logic lives in `dlm/`, so the laptop and Colab run identical code.

Run the cells top to bottom. Phases already finished on Drive are skipped (each phase is idempotent). **Runtime -> Change runtime type -> GPU** is needed only for pre-training (Phase 5, not built yet).


In [ ]:
# 1. Google Drive (data and the tokenizer persist here across Colab sessions)
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/diffusion_lm_outputs"


In [ ]:
# 2. Get the code (the GitHub repo must exist and be pushed; see CLAUDE.md "Main project folder")
REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
import os
if os.path.isdir("/content/dlm_repo"):
    !git -C /content/dlm_repo pull --ff-only
else:
    !git clone $REPO /content/dlm_repo
%cd /content/dlm_repo


In [ ]:
# 3. Dependencies. Do NOT pin or reinstall numpy/pillow here: it breaks Colab's live kernel.
!pip install -q -r requirements.txt


In [ ]:
# 4. Phase 0: download TinyStories (writes to Drive: diffusion_lm_outputs/data; a second run does nothing)
!python run.py phase0


In [ ]:
# 5. Phase 1: train + verify the tokenizer (writes to Drive; a second run does nothing)
!python run.py phase1 --out $OUT/tokenizer


In [ ]:
# 6. Phase 2: tokenize + block the Stories (writes data/tokens on Drive; a second run does nothing)
!python run.py phase2 --tokenizer-dir $OUT/tokenizer


## Phase 5: pre-training

Needs a **GPU with bf16** (L4 or A100); a T4 falls back to slow fp32. Runtime -> Change runtime type -> GPU first. Checkpoints and `final/` go to Drive (`diffusion_lm_outputs/checkpoints/pretrain_base`), so a disconnected session continues with `--resume`. The first evaluation lines show the loss leaving the ~5.9 token-frequency plateau (or not): watch it before leaving it running. Starting values come from the old notebooks' proxy trials and have never been run to completion on this model.


In [ ]:
# 7. Quick GPU check: a 200-step run of the tiny model (about a minute)
!python run.py pretrain --size tiny --steps 200 --eval-every 100 --warmup 20 --out $OUT/checkpoints/smoke --tokenizer-dir $OUT/tokenizer


In [ ]:
# 8. The real run (40,000 steps). Add --resume after a disconnect.
!python run.py pretrain --out $OUT/checkpoints/pretrain_base --tokenizer-dir $OUT/tokenizer


In [ ]:
# 9. Offline tests (optional)
!python run.py test


## Later phases (not built yet)

`python run.py generate` (Phase 6) and `python run.py sft` (Phase 8) will be added as cells here once they exist; see `PRD.md`.
